# Overview and Documentation of PQC Performance Study

## IT-Project 
**Performance of post-quantum cryptography (PQC)** <br>
summer semester 2025 - winter semester 2025/2026 <br>
Stefan, Shero, Jason, Alexander, Kristina <br>
Technische Hochschule Nürnberg Georg Simon Ohm

## Table of Contents
- [Abstract](#abstract)
- [Introduction](#introduction)
- [Background](#research)
  - [Quantum Algorithms](#algorithm)
  - [Quantum-Safe Cryptography](#pqc)
  - [Standards](#standards)
- [Methodology](#methodology)
  - [Libraries](#libraries)
    - [Liboqs](#liboqs)
    - [Botan](#botan)
    - [Extensibility and Customization](#custom)
  - [Experimental Environment](#environment)
  - [Limitations and Threats to Validity](#limitations)
- [Results](#results)
- [Additional Notebooks](#notebooks)
- [Conclusion](#conclusion)
- [References](#refs)

## Abstract <a id="abstract"></a>

Post-quantum cryptography (PQC) represents a critical advancement in cryptographic security designed to withstand attacks from both classical and quantum computing systems. Contemporary public‑key cryptography relies on the assumed intractability of problems such as integer factorization and the discrete logarithm. However, Shor’s algorithm enables polynomial‑time solutions to these problems on a cryptographically relevant quantum computer [2], rendering today’s public‑key schemes (e.g., RSA, ECC, Diffie–Hellman) vulnerable. In addition, “store‑now, decrypt‑later” adversary models increase the urgency of deployment, as data intercepted today may be decrypted once sufficient quantum capability becomes available. Security agencies have highlighted this risk since at least 2015, and the BSI assumes the availability of cryptographically relevant quantum computers in the early 2030s for high‑security planning horizons [1]. Against this backdrop, the development, standardization, and empirical evaluation of quantum‑resistant algorithms have become essential to ensure long‑term confidentiality and integrity in practical systems [4–7].

## Introduction <a id="introduction"></a>

This research project conducts an empirical evaluation of post‑quantum cryptographic (PQC) implementations with a focus on practical performance and deployability. We study representative Key‑Encapsulation Mechanisms (KEMs) and Digital Signature Algorithms (DSAs) from the NIST PQC process—primarily ML‑KEM (Kyber) and ML‑DSA (Dilithium), alongside selected non‑finalized or alternative candidates where instructive—across NIST security levels 1, 3, and 5. Using two mature libraries with complementary design goals, liboqs and Botan, we execute controlled measurements of key generation, encapsulation/signing, and decapsulation/verification times, as well as public‑/secret‑key, ciphertext, and signature sizes. A consistent experimental environment ensures comparability, and results are visualized to expose time–size trade‑offs.

The work addresses the following questions: What are the performance characteristics of standardized PQC algorithms at common security levels? How consistent are results across independent library implementations? Which algorithms offer favorable trade‑offs for bandwidth‑constrained, latency‑sensitive, or memory‑limited settings? And how do these observations inform near‑term migration strategies in real systems? 

This study aims to (i) characterize the performance and size profiles of standardized PQC KEMs and DSAs across NIST security levels, (ii) compare independent implementations to assess consistency and practical deployability, and (iii) derive guidance for algorithm selection under bandwidth, latency, and memory constraints together with near‑term migration considerations. Decision criteria emphasize consistency across libraries, robustness of results under repeated trials, and suitability for concrete constraints (certificate size, message size, handshake latency budgets, memory limits). The analysis is intended to translate raw measurements into practitioner‑oriented guidance, balancing security level targets with operational needs and migration feasibility.

Our contributions are threefold: a reproducible benchmarking suite (provided as notebooks for KEMs and signatures using liboqs and Botan), a consolidated and interpretable summary of empirical results, and guidance for algorithm selection and staged migration in typical deployment scenarios. Additionally, we provide a concrete, practical use case to illustrate the effective combination of PQC's KEM for secure key exchange with AES for high-speed data encryption. This scenario, the hybrid encryption of a PDF file, demonstrates how to leverage the strengths of both asymmetric (PQC) and symmetric cryptography to build a secure and performant system. The goal is to securely encapsulate a shared symmetric key for a recipient, which is then used to efficiently encrypt a large data file, such as a PDF.

The Introduction outlines scope, algorithms, libraries, and metrics; the following sections present the methodology and empirical results that address these objectives. Deliverables comprise: runnable benchmarking notebooks for KEMs and signatures (liboqs and Botan), aggregated tables and plots enabling quick comparison at each security level, and concise selection guidance summarizing recommended choices per deployment context and transition stage. Scope excludes cryptanalytic evaluation, formal security proofs, or side‑channel testing; where relevant, constant‑time claims and mitigations are noted from upstream documentation. The focus remains on empirically observable performance and footprint under controlled conditions to inform near‑term adoption decisions.

## Background  <a id="research"></a>

### Quantum algorithm <a id="algorithm"></a>

Quantum algorithms leverage the unique properties of qubits—the fundamental units of information in quantum computers—such as superposition and entanglement, to solve certain problems significantly faster than classical computers. While these algorithms can theoretically be simulated on classical machines, the exponential increase in computational effort for complex problems quickly makes such simulations impractical. <br>

Shor's Algorithm, developed in 1994, is critically important for cryptography. It can efficiently factor large numbers and solve the discrete logarithm problem—both mathematical foundations of many current public-key cryptosystems. Classical algorithms require sub-exponential time for these tasks, whereas Shor's achieves polynomial time complexity, potentially breaking current asymmetric cryptography. <br>

Grover's Algorithm, introduced by Lov Grover in 1996, offers a quadratic speedup for searching unsorted databases. It can find a target element in approximately  *$\sqrt{N}$* steps, compared to *N* steps for classical algorithms. This acceleration is significant for very large datasets and is relevant for the cryptanalysis of symmetric algorithms, for instance, by effectively halving the key length. However, its practical implementation still poses considerable technical challenges [1]. <br>

### Quantum-Safe Cryptography <a id= "pqc"></a>

To achieve quantum-resistance, PQC explores various approaches, which can be grouped into several fundamental categories. <br>
*Code-based cryptography* gets its security from the difficulty of efficiently decoding general error-correcting codes. A big advantage here is their high efficiency in encrypting and decrypting data, along with relatively small ciphertexts. However, a significant downside is the very large public key sizes, which can be in the megabyte range. <br>

*Lattice-based cryptography* relies on the difficulty of solving specific computational problems within mathematical lattices. These schemes are very efficient and make up a large portion of the candidates in the current NIST standardization process. Yet, the added mathematical structure also brings the risk of new potential attack methods. <br>

Finally, *hash-based cryptography* schemes leverage the strong properties of cryptographic hash functions for digital signatures. A key benefit of these schemes, is their mature security. Their main drawback, however, is statefulness: the signer must carefully track which one-time signature keys have been used. Any mistake here would compromise security and limit the total number of possible signatures. This puts high demands on how they're implemented and used [1].

### Standards <a id="standards"></a>

While multiple national bodies as well as companies pursue PQC, this work aligns with the NIST process due to its global impact and rigorous methodology [4]. For key establishment, CRYSTALS‑Kyber has been standardized as ML‑KEM (FIPS 203) [5]. For signatures, CRYSTALS‑Dilithium is standardized as ML‑DSA (FIPS 204) and SPHINCS+ as SLH‑DSA (FIPS 205) [6,7]. FALCON has been selected by NIST for standardization but is not finalized at the time of writing.

Our notebooks reflect this emphasis while also including additional schemes for context, depending on library support. In KEM benchmarks we prioritize ML‑KEM/“Kyber” across libraries and include HQC (liboqs), FrodoKEM (Botan), and Classic McEliece (Botan) as non‑standardized comparators. In signature benchmarks we measure ML‑DSA/Dilithium in both ecosystems; we include SPHINCS+ via liboqs (standardized as SLH‑DSA) and Falcon via liboqs (selected, not yet finalized); and we use Botan’s XMSS and HSS‑LMS to illustrate stateful hash‑based signatures. Where parameter sets map to specific NIST levels (e.g., ML‑DSA‑44 aligning to level 2), this is noted in the respective notebooks.

Accordingly, results and recommendations prioritize ML‑KEM, ML‑DSA, and SLH‑DSA, using the additional schemes to illuminate time–size trade‑offs across families and implementations.

## Methodology <a id="methodology"></a>

To ensure comparability across libraries and security levels, we applied a uniform measurement protocol in all notebooks. Each algorithm was evaluated at the NIST security levels 1, 3, and 5 using the parameter sets exposed by the respective library. We measured key generation, encapsulation or signing, and decapsulation or verification, and we recorded the sizes of public and secret keys, ciphertexts, and signatures. Measurements were executed with Python’s timeit over repeated runs, and we report minimum, average, and maximum values; where appropriate, plots use a logarithmic scale to make differences visible across orders of magnitude. To ensure the consistency and reproducibility of our results, all experiments were conducted within the same virtual machine. This environment was defined by Python dependencies precisely pinned in a `requirements.txt` file, and a custom-built Botan library was utilized for all cryptographic operations. 
For KEMs we validated correctness by checking that the shared secrets produced by sender and receiver match, and for signatures we verified that each produced signature validates against the corresponding public key. Any deviations necessitated by library interfaces—for example parameter naming or the availability of raw key export—are explicitly noted in the individual notebooks. 

### Libraries <a id="libraries"></a>

#### Liboqs <a id="liboqs"></a>

As a key open-source initiative facilitating the global transition to quantum-resistant cryptography, the Open Quantum Safe (OQS) project operates under the Linux Foundation. <br>

Its primary component, liboqs (Version 0.13.0), is an open-source C library that implements various quantum-resistant cryptographic algorithms. The project's versatility is a notable asset, providing language wrappers—including C++, Go, Java, Python, and Rust—to integrate these algorithms into diverse development environments. <br>

Our project specifically utilized the Python wrapper (Version 0.12.0), which proved to be exceptionally user-friendly, supported by comprehensive documentation and illustrative example code. Beyond its accessibility, OQS offers significant practical advantages, including extensive multi-platform compatibility demonstrated through testing on Linux, macOS, and Windows. Its continuous development ensures ongoing relevance and reliability. Furthermore, the documentation's inclusion of guidance on non-NIST-standardized algorithms is highly beneficial for research and evaluation purposes. The extensive selection of algorithms within OQS makes it an invaluable resource for exploring and comparing different PQC candidates, establishing it as a robust solution for PQC integration.

##### Troubleshooting 

**AttributeError: module 'oqs' has no attribute 'KeyEncapsulation'**

The `AttributeError: module 'oqs' has no attribute 'KeyEncapsulation'` is a common issue that occurs when the `liboqs` Python wrapper is either not installed correctly or is incompatible with the underlying C library. <br>
A simple reinstallation of the `liboqs-python` package, following the instructions on its official [GitHub repository](https://github.com/open-quantum-safe/liboqs-python), will usually resolve this error. This ensures that all necessary modules and dependencies are properly linked.

**MechanismNotEnabledError: HQC-128**

This error occurs because the HQC algorithm is not enabled by default in the current version of the underlying `liboqs` C library. To use HQC-128, you must compile the `liboqs` C library from source and explicitly enable the algorithm. You modify the CMake build options by including the flag `-DOQS_ENABLE_KEM_HQC=ON` when running CMake.

#### Botan <a id="botan"></a>

Botan 3 is a C++ cryptography library with Python bindings. Unlike liboqs, these bindings are not available via standard package managers such as pip. In our setup, Botan was compiled from source obtained from the project website, producing a local shared library (e.g., `botan-3.dll`, `libbotan-3.so.10`) and a Python module (e.g., `botan3.py`). To load it from Python, we added the DLL directory to the system PATH and ensured the generated Python module was on `sys.path` (as reflected in our notebooks via `ctypes.CDLL` and `sys.path.append`). This manual build and path configuration stands in clear contrast to liboqs’s streamlined installation and import workflow.

Within this project, we used the Python interface to benchmark Botan’s post‑quantum schemes in parity with our liboqs experiments. For KEMs, we evaluated Kyber/ML‑KEM where available and included FrodoKEM and Classic McEliece as non‑standardized comparators. For signatures, we measured Dilithium (mapped to ML‑DSA levels) and the stateful hash‑based schemes XMSS and HSS‑LMS; classical RSA, ECDSA, and Ed25519 were included for reference.

Botan’s position in the German ecosystem is noteworthy: its algorithm portfolio and parameterization align with guidance from the Bundesamt für Sicherheit in der Informationstechnik (BSI). While the BSI typically recommends standards and parameter sets rather than endorsing specific libraries, Botan’s support for ML‑KEM and ML‑DSA and its long maintenance history make it a sensible choice for projects aiming to follow BSI recommendations in practice.

Practical notes from our setup: on some platforms raw private key export is not supported, which can limit exact size reporting; availability of PQC algorithms may depend on build configuration and CPU features; as with liboqs, side‑channel aspects were out of scope for this work.

#### Extensibility and Customization <a id="custom"></a>

The developed benchmarking suite is designed with high extensibility and customization, allowing it to be flexibly used for future evaluations. This design ensures that new Post-Quantum Cryptography (PQC) algorithms or alternative implementations can be easily integrated into the measurements without fundamentally changing the existing codebase.

The modularity of the framework is based on a clear, step-by-step process. To evaluate a new library, its `import` statement first needs to be added to the script's requirements. The existing benchmark code can then be copied and adapted in a new file.

The algorithms to be tested are not hardcoded. Instead, they are dynamically defined via separate variables for each security level (e.g., `encryption_L1` or `signature_L3`). Since each library has its own named implementations, these lists must be specifically adapted for the new library.

Within the core functions of the benchmark (e.g., `generate_keys`, `encryption`, `decryption` for KEMs or `sign_message`, `verify` for signatures), the specific code for the new library is inserted. This allows the precise process for key generation, encryption/decryption, or signing/verification for each algorithm to be accurately represented.

This also includes adapting variables like `private_key` and `public_key` for key generation, `ciphertext` and `shared_secret_sender` for encryption, or `signature` for signing. The modular structure ensures that the rest of the benchmark logic, such as time measurement and output in tables and graphs, remains unchanged.

After the code has been adapted and the benchmarks executed, the results and the code are required to be documented in the corresponding Markdown cells. This documentation should include a precise formulation of the findings, along with specific notes, additional information, and a concluding summary based on the measurement results.

Thanks to this careful, modular architecture, the entire integration of a new algorithm can be completed in just a few, straightforward steps, which significantly accelerates the evaluation of new PQC standards and implementations.

### Experimental Environment <a id="environment"></a>

All experiments were executed in a dedicated virtual machine to ensure reproducible conditions across runs and libraries. The VM ran Ubuntu 25.04 under VirtualBox (version 7.1.4) and was configured with 6 GB of RAM, 2 vCPUs, and a 50 GB virtual disk. Running all benchmarks in the same guest environment minimizes host‑specific effects and reduces variance due to background activity or driver differences, thereby improving the comparability of absolute timings. Unless otherwise noted, the same VM snapshot was used for both the Botan and liboqs measurements.

### Limitations and Threats to Validity <a id="limitations"></a>

This study is limited to algorithmic runtime and size measurements; it does not assess cryptanalytic strength or side‑channel resilience. Absolute timings can vary with build options, CPU features, and operating systems; the fixed VM mitigates, but does not eliminate, such variance. Library coverage also differs—for example, certain parameter sets or raw key export may be unavailable—which occasionally prevents perfectly like‑for‑like comparisons; these cases are called out where relevant. Finally, the results do not directly translate to full protocol performance. End‑to‑end behavior in, for instance, a TLS handshake depends on networking conditions, I/O, certificate chain sizes, and application integration, all of which can materially affect latency and bandwidth.


## Results <a id="results"></a>

Across both libraries and all security levels, ML‑KEM (Kyber) consistently exhibits favorable performance and moderate key/ciphertext sizes relative to non‑standardized alternatives (e.g., FrodoKEM, Classic McEliece). For signatures, Dilithium (ML‑DSA) offers strong total time characteristics, while SPHINCS+ trades markedly higher signing/verification times for smaller keys and different security assumptions; Falcon performs competitively in many scenarios but is not finalized.

The cross‑library comparison shows broadly consistent trends, though absolute timings vary with implementation details and build options. These empirical observations support prioritizing standardized schemes—ML‑KEM for KEM and ML‑DSA/SLH‑DSA for signatures—while recognizing deployment‑specific constraints (bandwidth, latency budgets, memory) that may shift the optimal choice.

In order to get a bigger picture of the Results of the research, it is highly suggested to look into the individual Notebooks.


## Additional Notebooks <a id="notebooks"></a>
Execute the following code to install the libraries and access links to our related notebooks.

In [4]:
import sys
!"{sys.executable}" -m pip install -q -r requirements.txt

In [5]:
from IPython.display import Markdown

# Change your directory
# Adjust the base URL/path to your environment as needed
base_url = "http://localhost:8888/notebooks/PQC/"

notebooks = [
    # liboqs notebooks
    ("liboqs_KEM_TimeTests.ipynb", "liboqs KEM benchmarks"),
    ("liboqs_Sig_TimeTests.ipynb", "liboqs signature benchmarks"),
    # Botan notebooks
    ("Botan_KEM_TimeTests.ipynb", "Botan KEM benchmarks"),
    ("Botan_Sig_TimeTests.ipynb", "Botan signature benchmarks"),
   # use cases
   ("kem_encrypt_decrypt.ipynb", "Encryption of a PDF file with KEM and AES"),
    ("Signatur_demo.ipynb", "Verify Signature")
]

for filename, label in notebooks:
    link = f"{base_url}{filename}"
    display(Markdown(f"- [{label}]({link})"))

- [liboqs KEM benchmarks](http://localhost:8888/notebooks/PQC/liboqs_KEM_TimeTests.ipynb)

- [liboqs signature benchmarks](http://localhost:8888/notebooks/PQC/liboqs_Sig_TimeTests.ipynb)

- [Botan KEM benchmarks](http://localhost:8888/notebooks/PQC/Botan_KEM_TimeTests.ipynb)

- [Botan signature benchmarks](http://localhost:8888/notebooks/PQC/Botan_Sig_TimeTests.ipynb)

- [Encryption of a PDF file with KEM and AES](http://localhost:8888/notebooks/PQC/kem_encrypt_decrypt.ipynb)

- [Verify Signature](http://localhost:8888/notebooks/PQC/Signatur_demo.ipynb)

## Conclusion <a id="conclusion"></a>

This study set out to evaluate the practical performance and deployability of standardized post‑quantum cryptographic schemes across independent libraries and security levels. The results consistently favor ML‑KEM for key establishment due to its balanced combination of runtime and size, and they identify ML‑DSA as a compelling default for signatures, with SLH‑DSA serving as a standards‑compliant alternative under different assumptions. Falcon exhibits attractive performance but remains selected rather than finalized, and SPHINCS+ demonstrates that smaller key material may come at the cost of substantially higher operation times. Non‑standardized KEMs such as FrodoKEM and Classic McEliece provide useful context for understanding the time–size spectrum but do not challenge the standardized lattice‑based choices in typical deployment scenarios.

Beyond raw measurements, the cross‑library perspective matters: implementations differ in maturity, parameter exposure, and build configurations, and these differences translate into absolute timing variations. Nevertheless, the qualitative trends are stable. For practitioners, the central takeaway is to plan migrations around ML‑KEM and ML‑DSA, validate performance in representative environments, and budget for size and latency impacts at the protocol level, particularly for certificate chains and handshake flows. Where bandwidth is constrained or stateful operation is acceptable, SLH‑DSA and hash‑based schemes can be viable; where ultra‑low latency is paramount, careful profiling of Falcon and platform‑specific builds is warranted.

In short, standardized PQC is sufficiently mature for phased adoption. With a reproducible benchmarking setup, consistent environment, and transparent reporting of trade‑offs, organizations can make informed selections that balance security targets with operational realities and regulatory guidance, including the recommendations of the BSI. The accompanying notebooks, procedures, and presentation slides are intended to support such evidence-based decisions and to serve as a baseline for future reevaluation as implementations evolve.

## References <a id="refs"></a>

[1] Bundesamt für Sicherheit in der Informationstechnik (BSI). Kryptografie quantensicher gestalten. Available at: https://www.bsi.bund.de/SharedDocs/Downloads/DE/BSI/Publikationen/Broschueren/Kryptografie-quantensicher-gestalten.pdf <br>
[2] P. W. Shor. Algorithms for Quantum Computation: Discrete Logarithms and Factoring. In: Proc. 35th Annual Symposium on Foundations of Computer Science (FOCS), 1994. <br>
[3] L. K. Grover. A Fast Quantum Mechanical Algorithm for Database Search. In: Proc. 28th Annual ACM Symposium on Theory of Computing (STOC), 1996. <br>
[4] NIST PQC Project: Selected Algorithms. Available at: https://csrc.nist.gov/Projects/post-quantum-cryptography/selected-algorithms <br>
[5] NIST FIPS 203: Module‑Lattice‑Based Key‑Encapsulation Mechanism (ML‑KEM). DOI: https://doi.org/10.6028/NIST.FIPS.203 <br>
[6] NIST FIPS 204: Module‑Lattice‑Based Digital Signature (ML‑DSA). DOI: https://doi.org/10.6028/NIST.FIPS.204 <br>
[7] NIST FIPS 205: Stateless Hash‑Based Digital Signature (SLH‑DSA). DOI: https://doi.org/10.6028/NIST.FIPS.205 <br> 
[8] Open Quantum Safe (liboqs) project. Available at: https://openquantumsafe.org/ <br>
[9] liboqs algorithms documentation. Available at: https://openquantumsafe.org/liboqs/algorithms <br>
[10] Botan handbook. Available at: https://botan.randombit.net/handbook/ <br>
[11] Botan API reference. Available at: https://botan.randombit.net/doxygen/index.html <br>
